# 02B Incremental Append Pipeline

This `02_pipeline` variant demonstrates one clear pattern: **incremental driving source + full supporting source → append target**. FabricOps resolves incremental progress from the last Source Observation successfully committed for that exact source → target relationship.

On the first run, the missing committed baseline produces a deterministic complete-source bootstrap. Later runs return only the unconsumed watermark or changed-partition scope. A populated append target without committed source → target state fails safely rather than duplicating the complete source.

## Tested with FabricOps

This template has not yet been manually validated in Microsoft Fabric. Do not add a release to a Fabric-tested table until that validation has actually run.

# 0. Environment

Run the shared Fabric configuration and import the public APIs used by this pipeline variant.

In [ ]:
%run 00_env_config

In [ ]:
from pyspark.sql import functions as F

from fabricops_kit import (
    check_dq,
    check_freshness,
    check_guardrail_coverage,
    check_schema,
    check_sensitive_data,
    check_source_drift,
    pipeline_read,
    pipeline_write,
    profile_table,
    resolve_table_id,
    widget_select_data_contract,
)

# 1. Data Contract

Select the Data Contracts to test with this pipeline. Production automatically uses activated Data Contracts. This variant keeps `read_mode="incremental"` explicit and publishes with append.

In [ ]:
CONTRACTS = widget_select_data_contract(spark_session=spark)

# 2. Incremental → Append — Curated Orders

Define the append target first, then resolve each source relative to it. Orders is the incremental driving source; Products is a full supporting source.

In [ ]:
# 2.1 Target definition — identity must exist before an incremental read.
TARGET_1_STORE = "Silver"
TARGET_1_SCHEMA = "demo"
TARGET_1_TABLE = "curated_orders"
TARGET_1_LOAD_STRATEGY = "append"
target_1_table_id = resolve_table_id(
    store=TARGET_1_STORE,
    schema=TARGET_1_SCHEMA,
    table_name=TARGET_1_TABLE,
)

In [ ]:
# 2.2 Read Orders incrementally for Target 1.
# FabricOps resolves the committed source → target baseline; do not query metadata manually.
orders_1 = pipeline_read(
    store="Bronze",
    schema="demo",
    table_name="orders",
    read_mode="incremental",
    target_table_id=target_1_table_id,
    spark_session=spark,
)
orders_1_df = orders_1["dataframe"]

# Incremental batches are partial and must not replace the canonical complete-source profile.
# Freshness and Source Drift use the complete physical Source Observation captured by pipeline_read().
check_freshness(orders_1["table_id"])
check_schema(orders_1_df, table_id=orders_1["table_id"])
orders_1_dq = check_dq(orders_1_df, table_id=orders_1["table_id"])
check_source_drift(
    orders_1["table_id"],
    target_table_id=target_1_table_id,
    raise_on_failure=True,
)

In [ ]:
# 2.3 Read Products in full for the same target flow.
products_1 = pipeline_read(
    store="Bronze",
    schema="demo",
    table_name="products",
    read_mode="full",
    target_table_id=target_1_table_id,
    spark_session=spark,
)
products_1_df = products_1["dataframe"]
check_freshness(products_1["table_id"])
check_schema(products_1_df, table_id=products_1["table_id"])
products_1_dq = check_dq(products_1_df, table_id=products_1["table_id"])
# A full read remains eligible for canonical complete-source profiling.
products_1_profile = profile_table(store="Bronze", schema="demo", table_name="products")
check_source_drift(
    products_1["table_id"],
    target_table_id=target_1_table_id,
    raise_on_failure=True,
)

In [ ]:
# 2.4–2.7 Transform, check, write, then profile the complete persisted target.
# No-new-data is an explicit safe skip: no physical write and no progress commit occurs.
if orders_1["should_process"]:
    target_1_df = (
        orders_1_dq.get("dataframe", orders_1_df)
        .join(products_1_dq.get("dataframe", products_1_df), on="product_id", how="left")
        .withColumn("processed_at", F.current_timestamp())
    )
    check_schema(target_1_df, table_id=target_1_table_id)
    sensitive_1 = check_sensitive_data(target_1_df, table_id=target_1_table_id)
    target_1_dq = check_dq(sensitive_1["dataframe"], table_id=target_1_table_id)
    check_guardrail_coverage(
        target_table_id=target_1_table_id,
        source_table_ids=[orders_1["table_id"], products_1["table_id"]],
    )
    target_1_write = pipeline_write(
        target_1_dq.get("dataframe", sensitive_1["dataframe"]),
        store=TARGET_1_STORE,
        schema=TARGET_1_SCHEMA,
        table_name=TARGET_1_TABLE,
        load_strategy=TARGET_1_LOAD_STRATEGY,
        source_table_ids=[orders_1["table_id"], products_1["table_id"]],
        spark_session=spark,
    )
    # Read-back profiling preserves the canonical complete-table profile meaning.
    target_1_profile = profile_table(table_id=target_1_write["table_id"])
else:
    print("Target 1 → no unconsumed Orders data; publication and progress commit skipped.")

## Pattern boundary

This variant intentionally demonstrates **incremental read → append** only. Use a separate `02C` or `02D` pipeline variant for SCD patterns rather than mixing write strategies into this template.